In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
import requests
import datetime
from selenium import webdriver
from time import sleep
import os
from selenium.webdriver.chrome.service import Service as ChromeService


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'SC SFSA' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
#writer = ExcelWriter(filename)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running SC SFSA Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [4]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        regulatorName + ' 1': 'https://fsaseychelles.sc/regulated-entities/fiduciary',
        regulatorName + ' 2': 'https://fsaseychelles.sc/regulated-entities/hire-purchase-and-credit-sales',
        regulatorName + ' 3': 'https://fsaseychelles.sc/regulated-entities/insurance',
        regulatorName + ' 4': 'https://fsaseychelles.sc/regulated-entities/collective-investment-scheme',
        regulatorName + ' 5': 'https://fsaseychelles.sc/regulated-entities/capital-markets',
        regulatorName + ' 6': 'https://fsaseychelles.sc/regulated-entities/regulatory-sandbox',

        }



Typology={

       regulatorName + ' 1': 'Fiduciary',
       regulatorName + ' 2': 'Hire Purchase and Credit Sales',
       regulatorName + ' 3': 'Insurance',
       regulatorName + ' 4': 'Collective Investment Scheme',
       regulatorName + ' 5': 'Capital Markets',
       regulatorName + ' 6': 'Regulatory Sandbox',
        }


block_list = {
    regulatorName + ' 3': ['tab-insurance-agents', 'tab-insurance-sub-agents','tab-principal-insurance-representative'], 
    regulatorName + ' 5': ['tab-securities-dealer-representative', 'tab-investment-advisor-representative'], 
     
    }

In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict


In [6]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"
    }

    resp = requests.get(regdict[reg], headers=headers, timeout=30, verify=False)
    resp.raise_for_status()
    print(resp.url)
    print(resp.status_code)
    soup = BeautifulSoup(resp.text, "html.parser")
    tab_nums_ = soup.find_all('li', class_='nav-item')


    for tab_ in tab_nums_:
        if reg in block_list and tab_.find('a')['href'].replace('#','') in block_list[reg]:
            print(f"[INFO] : Skipping {tab_.find('a')['href'].replace('#','')} as per block list")
            continue
        data_url = regdict[reg] + tab_.find('a')['href']
        print(data_url)
        resp_tab = requests.get(data_url, headers=headers, timeout=30, verify=False)
        soup_tab = BeautifulSoup(resp_tab.text, "html.parser")
        tab_contents = soup_tab.find_all('div', class_='tab-content')
        for tab_content in tab_contents:
            tab_panes = tab_content.find('div', id=tab_.find('a')['href'].replace('#',''))
            if tab_panes:
                cards = tab_panes.find_all('div',class_='card')
                for card in cards:
                    card_title_ = card.find('h5').text.strip()
                    card_body_ = card.find('div', class_='card-body')
                    print(card_title_)
                    map_icon = card_body_.select_one("span.fa-map")
                    #print(map_icon)
                    address_lines = []
                    if map_icon:
                        for sib in map_icon.next_siblings:
                            if getattr(sib, "name", None) == "p":
                                break
                            text = getattr(sib, "get_text", lambda **_: str(sib))().strip()
                            if text:
                                address_lines.append(text)
                    address = " ".join(address_lines)

                    def text_after_icon(icon_class,root):
                        icon = root.select_one(f"span.{icon_class}")
                        if not icon:
                            return ''
                        a = icon.find_next("a")
                        return a.get_text(strip=True) if a else ''
                    phone = text_after_icon("fa-phone",card_body_)
                    mobile = text_after_icon("fa-mobile",card_body_)
                    fax = text_after_icon("fa-fax",card_body_)
                    email = text_after_icon("fa-envelope",card_body_)
                    website = text_after_icon("fa-globe",card_body_)

                    print(address)

                    sqldict['Name'].append(card_title_)
                    sqldict['Address_1'].append(address)
                    sqldict['Phone'].append(phone)
                    sqldict['Fax'].append(fax)
                    sqldict['Website'].append(website)
                    sqldict['Email'].append(email)
                    sqldict['ListName'].append(Typology[reg])
                    sqldict['Cntry'].append('SC')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['Typology'].append(tab_.text.strip())
                    sqldict = bourange_same_length_array(sqldict)



[INFO] : Working 1/6 _(SC SFSA 1)_ 


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fsaseychelles.sc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://fsaseychelles.sc/regulated-entities/fiduciary
200
https://fsaseychelles.sc/regulated-entities/fiduciary#tab-icsp


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fsaseychelles.sc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


A.C. Management Limited
Suite 10, 3rd Floor, La Ciotat, Mont Fleuri, Mahe, Seychelles
AAA International Services Ltd
House of Francis, Room 303, Ile Du Port, MahÃ©, SEYCHELLES
Acclime Seychelles Limited
303 Aarti Chambers, P.O. Box 983, Victoria, MahÃ©, SEYCHELLES
All About Offshore (Seychelles) Limited
1st Floor, Suite 15, Oliaji Trade Centre, Rue Pierre De Possession, Victoria, Mahe, Seychelles
Apex Corporate Services (Seychelles) Ltd
Suite 202, Second Floor, Eden Plaza, Eden Island, Mahe, Republic of Seychelles
Appleby Global Services (Seychelles) Limited
Suite 202, 2nd Floor, Eden Plaza, Eden Island, P O Box 1352 MahÃ©, SEYCHELLES
Axis Fiduciary (Seychelles) Limited
F20, 1st Floor, Eden Plaza, Eden Island
CARRÃ‰ TRUST (SEYCHELLES) LIMITED
Carre Chambers, Suite 104, Capital City Building, Victoria, Mahe, Seychelles
Cititrust (Seychelles) International Limited
Mont Fleuri, MahÃ©, SEYCHELLES
Crystal (Seychelles) Limited
Oliaji Trade Centre, P.O. Box 1101, Victoria, MahÃ©, SEYCHELLES
E

...

CABOTO INVESTMENTS LTD
CABOTO INVESTMENTS LTD Room B11, 1st Floor, Providence Complex, Mahe Seychelles
Peter Pesic & Co Securities (Seychelles) Limited
Overseas Contact: Ms. Fabiana Alves Siqueira Pesic Phone: +230 58152474 Email: fabianapesic@gmail.com
[INFO] : Working 6/6 _(SC SFSA 6)_ 


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fsaseychelles.sc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://fsaseychelles.sc/regulated-entities/regulatory-sandbox
200
https://fsaseychelles.sc/regulated-entities/regulatory-sandbox#tab-regulatory-sandbox


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'fsaseychelles.sc'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


SECDEX Digital Custodian Limited
Room No. F2-1, Second Floor, Oceanic Motors Building, Providence, MahÃ©, Seychelles


In [7]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)

In [11]:
df = df.drop_duplicates(subset=['Name','ListCode'])

In [ ]:

import json
import requests
API_TOKEN = ""
URL = "https://Api.bvdinfo.com/v1/orbis/companies/match"



headers = {
    "ApiToken": API_TOKEN,
    "Content-Type": "application/json"
}
# df is your dataframe with a "Name" column
for i, row in df.iterrows():
    ITERATE_NAME = str(row["Name"]).strip() if row["Name"] is not None else ""
    fax_ = str(row["Fax"]).strip() if row["Fax"] is not None else ""
    tel_ = str(row["Phone"]).strip() if row["Phone"] is not None else ""
    address_ = str(row["Address_1"]).strip() if row["Address_1"] is not None else ""
    email_ = str(row["Email"]).strip() if row["Email"] is not None else ""
    website_ = str(row["Website"]).strip() if row["Website"] is not None else ""
    payload = {
        "MATCH": {
            "Criteria": {
                "Name": ITERATE_NAME,
                "Country": "SC",
                "EMailOrWebsite": email_ or website_,
                "Address": address_,
                "PhoneOrFax": fax_ or tel_
            },
            "Options": {
                "ScoreLimit": 0.85,
                "ExclusionFlags": ["ExcludeBranchLocations","ExcludeHistorical","ExcludeInactive"]
            }
        },
        "SELECT": [
            "Match.Hint",
            "Match.Score",
            "Match.Name",
            "Match.Name_Local",
            "Match.Address",
            
            "Match.Postcode",
            "Match.City",
            "Match.Country",
            "Match.Status",
            "Match.National_Id",
            "Match.NationalIdLabel",
            "Match.LegalForm",
            "Match.BvDId",
            "Match.PhoneOrFax",
            "Match.EmailOrWebsite"
        ]
    }

    response = requests.post(URL, headers=headers, json=payload, timeout=60)
    #print("Status:", response.status_code)

    try:
        data = response.json()
        bvd_id = ""
        print(json.dumps(data, indent=2))
        if data and data[0].get("Hint", "").lower() != "unlikely":
            if len(data) > 1:
                print(f"Multiple matches found for row {i}: {len(data)} matches")
                top_score = data[0].get("Score", 0)
                tied = [d for d in data if d.get("Score", 0) == top_score]

                if len(tied) > 1:
                    words = [w for w in ITERATE_NAME.split() if w]
                    def hint_name_count(d):
                        name = d.get("Name", "").lower()
                        return sum(1 for w in words if w.lower() in name)

                    best = max(tied, key=hint_name_count)  # first wins on tie
                    print(best)
                    data[0] = best


            bvd_id = data[0].get("BvDId", "")
    except Exception:
        #print(response.text)
        bvd_id = ""
    #print(bvd_id)
    df.at[i, "bvdid"] = bvd_id if bvd_id else ""



[
  {
    "Hint": "Selected",
    "Score": 0.99,
    "Name": "A.C. MANAGEMENT LIMITED",
    "Name_Local": null,
    "Address": "SUITE 10, 3RD FLOOR, LA CIOTAT, MONT FLEURI",
    "Postcode": null,
    "City": "MAHE",
    "Country": "SC",
    "Status": "Active",
    "National_Id": "C8410125",
    "NationalIdLabel": "Business Registration Number",
    "LegalForm": "Private limited company",
    "BvDId": "SCC8410125",
    "PhoneOrFax": "+248 4325886",
    "EmailOrWebsite": "www.acmanagementltd.com"
  }
]
[
  {
    "Hint": "Selected",
    "Score": 0.99,
    "Name": "AAA INTERNATIONAL SERVICES LTD",
    "Name_Local": null,
    "Address": "HOUSE OF FRANCIS, ROOM 303, ILE DU PORT",
    "Postcode": null,
    "City": "VICTORIA",
    "Country": "SC",
    "Status": "Active",
    "National_Id": "C844935",
    "NationalIdLabel": "Business Registration Number",
    "LegalForm": "Private limited company",
    "BvDId": "SCC844935",
    "PhoneOrFax": "+248 4374300",
    "EmailOrWebsite": "info@aaaintern